In [ ]:
!pip install -q git+https://github.com/huggingface/diffusers
!pip install -q transformers accelerate gradio torch

In [ ]:
import torch
from diffusers import DiffusionPipeline
import gradio as gr

# -----------------------------
# Model Setup (Load once)
# -----------------------------
model_name = "Qwen/Qwen-Image"

if torch.cuda.is_available():
    torch_dtype = torch.bfloat16
    device = "cuda"
else:
    torch_dtype = torch.float32
    device = "cpu"

pipe = DiffusionPipeline.from_pretrained(model_name, torch_dtype=torch_dtype)
pipe = pipe.to(device)

In [ ]:
positive_magic = {
    "en": ", Ultra HD, 4K, cinematic composition.",
    "zh": ", 超清，4K，电影级构图."
}

aspect_ratios = {
    "1:1": (1328, 1328),
    "16:9": (1664, 928),
    "9:16": (928, 1664),
    "4:3": (1472, 1140),
    "3:4": (1140, 1472),
    "3:2": (1584, 1056),
    "2:3": (1056, 1584),
}

In [ ]:
# -----------------------------
# Inference Function
# -----------------------------
def generate_image(prompt, aspect_ratio, steps, cfg_scale, seed):
    if not prompt.strip():
        return None, "❌ Please enter a prompt."

    width, height = aspect_ratios[aspect_ratio]

    # Handle seed
    if seed == -1:
        seed = torch.seed()

    generator = torch.Generator(device=device).manual_seed(int(seed))

    try:
        image = pipe(
            prompt=prompt + positive_magic["en"],
            negative_prompt="",
            width=width,
            height=height,
            num_inference_steps=steps,
            true_cfg_scale=cfg_scale,
            generator=generator
        ).images[0]

        return image, f"✅ Done | Seed: {seed}"

    except Exception as e:
        return None, f"❌ Error: {str(e)}"

In [ ]:
# -----------------------------
# UI
# -----------------------------
with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("# 🎨 Qwen Image Generator (Colab)")

    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label="Prompt",
                placeholder="Describe your image...",
                lines=4
            )

            aspect_dropdown = gr.Dropdown(
                choices=list(aspect_ratios.keys()),
                value="16:9",
                label="Aspect Ratio"
            )

            steps_slider = gr.Slider(10, 100, value=50, step=1, label="Steps")
            cfg_slider = gr.Slider(1.0, 10.0, value=4.0, step=0.5, label="CFG Scale")

            seed_input = gr.Number(value=42, label="Seed (-1 = random)")

            generate_btn = gr.Button("🚀 Generate")

        with gr.Column():
            output_image = gr.Image(label="Output")
            status_text = gr.Textbox(label="Status")

    generate_btn.click(
        fn=generate_image,
        inputs=[prompt_input, aspect_dropdown, steps_slider, cfg_slider, seed_input],
        outputs=[output_image, status_text]
    )

In [ ]:
# IMPORTANT FOR COLAB
app.queue()
app.launch(share=True)